## LangChain day - 7 : Text Splitters

Text Splitting is the process of breaking large chunks of text (like articles, PDFs, HTML pages, or books) into smaller, manageable pieces (chunks) that a Large Language Model (LLM) can handle effectively.

Text splitting is a crucial process in working with Large Language Models (LLMs) for several key reasons:

### Overcoming Model Limitations
Many embedding models and LLMs have **maximum input size constraints**. If a document is too long, it simply cannot be fed into the model in its entirety. Text splitting breaks down these large documents into smaller, manageable **chunks** that fit within the model's limits. This allows you to process documents that would otherwise exceed these boundaries.

### Improving Downstream Tasks
Text splitting significantly **improves nearly every LLM-powered task** by making the input more focused and digestible.

| Task           | Why Splitting Helps                                                               |
| :------------- | :-------------------------------------------------------------------------------- |
| **Embedding** | Short chunks yield **more accurate vectors** (numerical representations of text), as the embeddings capture the meaning of a smaller, more coherent piece of text. |
| **Semantic Search** | Search results point to **focused information**, reducing "noise" and improving relevance. When you search, you get back specific, relevant chunks rather than entire, unwieldy documents. |
| **Summarization** | Prevents **hallucination and topic drift**. By summarizing smaller, distinct chunks, the LLM is less likely to generate irrelevant or incorrect information, staying true to the source content of each chunk. |

### Optimizing Computational Resources
Working with smaller chunks of text is generally **more memory-efficient** and allows for **better parallelization** of processing tasks. This means that operations like embedding or feeding data to an LLM can be done more quickly and with less computational overhead, especially when dealing with very large datasets.

### Type of text Splitters :
1. Length Based
2. Text Structure Based
3. Document Structure Based
4. Semantic Meaning Based

### 1\. Length-Based Splitter

This splitter is a brute-force method that divides text into chunks of a predetermined size, without any regard for grammar or content.

  * **Original Text:**

    ```
    "The first machine learning models were quite simple. Early algorithms like linear regression and logistic regression laid the groundwork for more complex systems. Today, we see sophisticated neural networks and transformers, but these foundational methods remain crucial for understanding the core principles of AI."
    ```

  * **Simulation (Chunk Size = 100 characters):**
    The splitter starts at the beginning of the text and simply counts characters, making a cut every 100 characters.

      * **Chunk 1 (0-100):** `"The first machine learning models were quite simple. Early algorithms like linear regression a"`
      * **Chunk 2 (101-200):** `"nd logistic regression laid the groundwork for more complex systems. Today, we see sophistica"`
      * **Chunk 3 (201-282):** `"ted neural networks and transformers, but these foundational methods remain crucial for unders"`
      * **Chunk 4 (283-360):** `"tanding the core principles of AI."`

  * **Problem:** As you can see, this method creates chunks that are difficult to read and completely break the flow of sentences and even words (e.g., "regression a" and "sophistica"). The chunks are purely based on length, making them semantically incoherent.

-----

### 2\. Text Structure-Based Splitter

This splitter is a more intelligent approach that uses a list of characters as delimiters to split the text. It tries to split on the most significant delimiter first, ensuring that it keeps sentences or paragraphs together whenever possible.

  * **Original Text:**

    ```
    "Deep learning is a subset of machine learning. It uses multi-layered neural networks to achieve state-of-the-art results in many fields.

    Unlike traditional machine learning, which often relies on human-designed features, deep learning models can learn features directly from the data. This has led to major breakthroughs in computer vision and natural language processing."
    ```

  * **Simulation (Delimiters = `['\n\n', '.', ' ']`):**
    The splitter will try to find the `'\n\n'` (double newline) first. This is a good way to identify logical paragraph breaks.

      * **Step 1:** The splitter finds the `'\n\n'` and splits the text into two main parts.

          * **Chunk A:** `"Deep learning is a subset of machine learning. It uses multi-layered neural networks to achieve state-of-the-art results in many fields."`
          * **Chunk B:** `"Unlike traditional machine learning, which often relies on human-designed features, deep learning models can learn features directly from the data. This has led to major breakthroughs in computer vision and natural language processing."`

      * **Step 2:** The splitter then checks if either of these chunks is too long (you can set a `chunk_size`). If so, it moves to the next delimiter in the list, which is `'.'`. Since both chunks are relatively short, they would likely remain as is, preserving the full sentences. If Chunk B was much longer, the splitter might use the period to break it down further into individual sentences, like: `"Unlike traditional machine learning..."` and `"This has led to major breakthroughs..."`.

  * **Benefit:** This method prioritizes maintaining the integrity of paragraphs and sentences, making the resulting chunks far more readable and meaningful than with a length-based approach.

-----

### 3\. Document Structure-Based Splitter

This specialized splitter understands the syntax of structured documents like Markdown or HTML. It uses the document's own hierarchy (e.g., headings, lists) to create logical chunks.

  * **Original Text (Markdown format):**

    ```
    # Introduction to Generative AI

    Generative AI is a field of artificial intelligence focused on creating new content. This can include text, images, music, and more.

    ## Key Concepts
    * **Models:** Examples include GPT and DALL-E.
    * **Training:** These models are trained on vast datasets.

    ## Applications
    The technology is used for tasks such as:
    * Writing creative text
    * Generating realistic images
    * Developing new drug compounds
    ```

  * **Simulation:**
    The splitter recognizes the Markdown syntax. It sees the top-level heading (`#`) and the subheadings (`##`). It knows that content under a heading belongs to that heading's context.

      * **Chunk 1:**
        ```
        "Introduction to Generative AI

        Generative AI is a field of artificial intelligence focused on creating new content. This can include text, images, music, and more."
        ```
      * **Chunk 2:**
        ```
        "Key Concepts
        * **Models:** Examples include GPT and DALL-E.
        * **Training:** These models are trained on vast datasets."
        ```
      * **Chunk 3:**
        ```
        "Applications
        The technology is used for tasks such as:
        * Writing creative text
        * Generating realistic images
        * Developing new drug compounds"
        ```

  * **Benefit:** This method ensures that each chunk is a semantically complete section of the document, even if the individual parts are short. It's ideal for keeping a heading with its associated paragraphs or a list with its introductory sentence, preserving the document's original logical flow.

-----

### 4\. Semantic Meaning-Based Splitter

This is the most advanced method, using a language model to understand the *meaning* of the text. It breaks the document where the topic changes significantly, ensuring that each chunk is semantically coherent.

  * **Original Text:**

    ```
    "The transformer architecture was a major breakthrough in natural language processing (NLP). Introduced in 2017, its self-attention mechanism allowed models to weigh the importance of different words in a sentence, leading to powerful models like BERT and GPT.

    However, the immense computational cost of training these large transformer models has raised concerns. The energy consumption and environmental impact of running massive data centers for training are now a significant area of research. Additionally, there are ethical debates around bias and misuse.

    Looking to the future, researchers are exploring more efficient model architectures, such as mixture-of-experts (MoE) models, and developing techniques for more sustainable training. The goal is to democratize access to powerful AI while minimizing its negative externalities."
    ```

  * **Simulation:**

    1.  **Embedding Generation:** The splitter first breaks the text into sentences or small paragraphs. It then uses an embedding model to generate a vector representation for each of these small chunks. These vectors represent the semantic meaning of the text.
    2.  **Vector Distance Analysis:** The splitter then calculates the "distance" between the vectors of adjacent chunks. A small distance means the topics are similar. A large distance indicates a significant change in topic.
    3.  **Splitting:**
          * The distance between the first and second paragraph would be large because the topic shifts from **"technical breakthrough (transformer architecture)"** to **"challenges and ethical concerns (computational cost, environment, bias)."** The splitter would make a cut here.
          * The distance between the second and third paragraph would also be large because the topic shifts from **"problems and concerns"** to **"future solutions and research directions."** The splitter would make another cut here.

  * **Final Chunks:**

      * **Chunk 1 (History):** "The transformer architecture... BERT and GPT."
      * **Chunk 2 (Problems):** "However, the immense computational cost... bias and misuse."
      * **Chunk 3 (Future):** "Looking to the future... negative externalities."

  * **Benefit:** This method is highly effective for tasks like summarization or question-answering because it guarantees that each chunk contains a single, coherent topic, which is essential for providing relevant and accurate context to a language model.